In [ ]:
# Cell 1 - Imports

import os
import glob
import numpy as np
import pydicom
import scipy.ndimage
import matplotlib.pyplot as plt


In [ ]:
# Cell 2a – Explore available datasets
print("Top-level folders in /kaggle/input:")
print(os.listdir("/kaggle/input"))


In [ ]:
# Cell 2b – Inspect 'rsna-intracranial-aneurysm-detection' dataset
path_1 = "/kaggle/input/rsna-intracranial-aneurysm-detection"
print("Contents of rsna-intracranial-aneurysm-detection:")
print(os.listdir(path_1))


In [ ]:
# Cell 2c – Inspect contents of 'series' directory
series_path = os.path.join(path_1, "series")
print("Sample entries in series/:", os.listdir(series_path)[:5])


In [ ]:
# Cell 2d – Inspect contents of a sample series UID folder
sample_uid = os.listdir(series_path)[0]
sample_uid_path = os.path.join(series_path, sample_uid)

print("Inspecting:", sample_uid_path)
print("Sample files:", os.listdir(sample_uid_path)[:5])


In [ ]:
# Cell 2e – Define DICOM Loader Function

def load_dicom_series(dicom_folder):
    dicom_files = glob.glob(os.path.join(dicom_folder, "*.dcm"))
    dicoms = [pydicom.dcmread(f) for f in dicom_files]

    # Only keep DICOMs with necessary attributes
    dicoms = [d for d in dicoms if hasattr(d, "ImagePositionPatient") and hasattr(d, "SOPInstanceUID")]

    if len(dicoms) == 0:
        raise ValueError(f"No valid DICOMs with ImagePositionPatient found in {dicom_folder}")

    # Sort by Z-axis
    dicoms.sort(key=lambda x: float(x.ImagePositionPatient[2]))

    volume = np.stack([d.pixel_array for d in dicoms])
    spacing = tuple(map(float, dicoms[0].PixelSpacing)) + (float(dicoms[0].SliceThickness),)

    return volume.astype(np.int16), spacing


In [ ]:
import pandas as pd

# Adjust the path if your train.csv is stored elsewhere
labels_path = "/kaggle/input/rsna-intracranial-aneurysm-detection/train.csv"
labels_df = pd.read_csv(labels_path)

print("Labels DataFrame loaded:", labels_df.shape)


In [ ]:
# Cell 2f – Load Sample Volume

sample_row = labels_df.iloc[4250]  # or any valid row index
sample_uid = sample_row["SeriesInstanceUID"]
series_base = "/kaggle/input/rsna-intracranial-aneurysm-detection/series"

# ✅ Confirm folder exists and shows .dcm files
sample_uid_path = os.path.join(series_base, sample_uid)
print("Sample UID path:", sample_uid_path)
print("Contents of DICOM folder:", os.listdir(sample_uid_path))  # Should show .dcm files

# ✅ Now load volume
sample_vol, sample_spacing = load_dicom_series(sample_uid_path)

print("Loaded volume shape:", sample_vol.shape)
print("Voxel spacing (x, y, z):", sample_spacing)




In [ ]:
# Cell 3 – Load training labels
import pandas as pd

labels_df = pd.read_csv("/kaggle/input/rsna-intracranial-aneurysm-detection/train.csv")
print("Shape:", labels_df.shape)
labels_df.head()


In [ ]:
# Cell 3a – Look up label metadata for sample UID
sample_row = labels_df[labels_df["SeriesInstanceUID"] == sample_uid]

if sample_row.empty:
    print(f"UID {sample_uid} not found in train.csv")
else:
    display(sample_row)


In [ ]:
# Cell 4a – Setup output directory and helper imports
import os
import numpy as np

# Create directory for saving preprocessed volumes
output_dir = "/kaggle/working/processed_volumes"
os.makedirs(output_dir, exist_ok=True)

print(f"Output directory set: {output_dir}")


In [ ]:
# Cell 4b – Normalize and resample functions
import scipy.ndimage

def normalize_volume(volume, clip_bounds=(-1000, 1000)):
    volume = np.clip(volume, *clip_bounds)
    mean = np.mean(volume)
    std = np.std(volume)
    return (volume - mean) / std

def resample_volume(volume, spacing, new_spacing=(1.0, 1.0, 1.0)):
    resize_factor = np.array(spacing) / np.array(new_spacing)
    new_shape = np.round(volume.shape * resize_factor).astype(int)
    return scipy.ndimage.zoom(volume, resize_factor, order=1)


Load from DICOM UID folder

Normalize and (optionally) resample

Save .npy file to working directory

Store metadata for later DataFrame export

In [ ]:
# Cell 4c – Batch process and save
from tqdm import tqdm

uids = labels_df["SeriesInstanceUID"].tolist()
series_base = "/kaggle/input/rsna-intracranial-aneurysm-detection/series"

metadata_records = []

for uid in tqdm(uids[:20]):  # start with 20 for test run
    uid_path = os.path.join(series_base, uid)
    try:
        vol, spacing = load_dicom_series(uid_path)
        vol_norm = normalize_volume(vol)
        # Optional resampling:
        # vol_norm = resample_volume(vol_norm, spacing[::-1], new_spacing=(1,1,1))

        out_path = os.path.join(output_dir, f"{uid}.npy")
        np.save(out_path, vol_norm)

        row = labels_df[labels_df["SeriesInstanceUID"] == uid].iloc[0]
        label_present = int(row.iloc[4:].sum() > 0)

        metadata_records.append({
            "UID": uid,
            "Filename": f"{uid}.npy",
            "Shape": vol.shape,
            "Spacing": spacing,
            "Modality": row["Modality"],
            "PatientAge": row["PatientAge"],
            "PatientSex": row["PatientSex"],
            "AneurysmPresent": label_present
        })

    except Exception as e:
        print(f"Failed for UID {uid}: {e}")


In [ ]:
# Cell 4d – Save metadata
metadata_df = pd.DataFrame(metadata_records)
metadata_df.to_csv("/kaggle/working/volume_metadata.csv", index=False)

print("Saved metadata for processed volumes.")
display(metadata_df.head())


In [ ]:
# Cell 5 – Resample & Normalize (Pre-processing)

def resample_volume(volume, spacing, new_spacing=(1.0, 1.0, 1.0)):
    resize_factor = np.array(spacing) / np.array(new_spacing)
    new_shape = np.round(np.array(volume.shape) * resize_factor).astype(int)
    return scipy.ndimage.zoom(volume, resize_factor, order=1)

def normalize_volume(volume, clip_bounds=(-1000, 1000)):
    volume = np.clip(volume, *clip_bounds)
    mean, std = np.mean(volume), np.std(volume)
    return (volume - mean) / std



In [ ]:
# Cell 6 – Volume Slice Visualization

def show_volume_slices(volume, title="Volume"):
    mid_axial = volume[volume.shape[0] // 2]
    mid_coronal = volume[:, volume.shape[1] // 2, :]
    mid_sagittal = volume[:, :, volume.shape[2] // 2]
    
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(mid_axial, cmap='gray'); axs[0].set_title('Axial')
    axs[1].imshow(mid_coronal, cmap='gray'); axs[1].set_title('Coronal')
    axs[2].imshow(mid_sagittal, cmap='gray'); axs[2].set_title('Sagittal')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 7 – Load and Visualize CTA Volume

cta_path = os.path.join(patient_path, "CTA")

# Confirm folder exists
assert os.path.exists(cta_path), f"CTA path not found: {cta_path}"

cta_vol, cta_spacing = load_dicom_series(cta_path)
print(f"CTA volume shape: {cta_vol.shape}, spacing: {cta_spacing}")

cta_resampled = resample_volume(cta_vol, spacing=cta_spacing[::-1])  # ZYX → XYZ
cta_normalized = normalize_volume(cta_resampled)

show_volume_slices(cta_normalized, title=f"CTA – Patient {sample_patient}")
